# CD-HIT 蛋白序列关联样本元数据

在项目根目录运行本 Notebook。以 CD-HIT 的代表 FASTA 为输入，每条蛋白输出一行。
如果当前内核缺少 openpyxl，在代码单元中运行 %pip install openpyxl 后再执行本流程。

- **fasta_sample_id** 来自蛋白 ID，去掉末尾的七位基因编号。
- **s1_sample_id** 和 **s1_accession** 分别是补充表 S1 的样本编号和数据登录号。
- **s1_row** 是样本在原工作表中的行号，便于回查。
- 温度字段保留原表中的 N/A。

In [1]:
import csv
from itertools import zip_longest
import os
from pathlib import Path
import re
from tempfile import NamedTemporaryFile

from openpyxl import load_workbook

project = Path.cwd()
fasta_path = project / 'output/cdhit/meer1000M_cd90.pep.fa'
s1_path = project / 'dataset/MEER/DSGC_Table_S1.xlsx'
alias_path = project / 'dataset/论文附录数据/07_DSGC_CHM2026_TableS1_metagenome_metadata_latlon.xlsx'
output_path = fasta_path.with_name('meer1000M_cd90_DSGC_Table_S1.csv')

## 1. 建立样本对应关系

优先匹配原样本 ID，其次统一下划线与连字符并检查对应关系唯一。
FDZ064-BuG10-12-7 根据两版 S1 第 420 行对应到 FDZ064-BuG10-12。
两行的经纬度、深度、材料和总测序量一致。导出的登录号和元数据取自 DSGC_Table_S1。

In [2]:
def read_s1(path):
    workbook = load_workbook(path, read_only=True, data_only=True)
    try:
        rows = workbook['S1'].iter_rows(min_row=4, values_only=True)
        return {n: row for n, row in enumerate(rows, start=4) if row[1]}
    finally:
        workbook.close()


def sample_fields(cell):
    stem, separator, accession = cell.strip().rpartition(' (')
    if separator and accession.endswith(')'):
        return stem.strip(), accession[:-1]
    return cell.strip(), ''


def normalize_sample_id(sample_id):
    return re.sub(r'[-_]+', '-', sample_id).strip('-')


s1_rows = read_s1(s1_path)
samples = {}
normalized_samples = {}
for row_number, row in s1_rows.items():
    sample_id, accession = sample_fields(row[1])
    normalized = normalize_sample_id(sample_id)
    if sample_id in samples or normalized in normalized_samples:
        raise ValueError(f'S1 样本编号不唯一：{sample_id}')
    samples[sample_id] = (row_number, accession, row)
    normalized_samples[normalized] = sample_id

original_row = s1_rows[420]
alias_row = read_s1(alias_path)[420]
original_id, _ = sample_fields(original_row[1])
alias_id, _ = sample_fields(alias_row[1])
if (original_id, alias_id) != ('FDZ064-BuG10-12', 'FDZ064-BuG10-12-7'):
    raise ValueError('两版 S1 第 420 行的样本编号有变动')
# 两版 S1 使用不同登录号。核对这两列之外的样本元数据。
for i, (original, revised) in enumerate(zip(original_row, alias_row)):
    if i not in (1, 9) and original != revised:
        raise ValueError('两版 S1 第 420 行的样本元数据不一致')


def lookup_sample(sample_id):
    if sample_id in samples:
        return sample_id
    canonical = normalized_samples.get(normalize_sample_id(sample_id))
    if canonical is not None:
        return canonical
    if sample_id == alias_id:
        return original_id
    raise ValueError(f'找不到样本元数据：{sample_id}')

## 2. 拼接代表 FASTA 与元数据

按 FASTA 原顺序输出，每条氨基酸序列拼成一列。

In [3]:
columns = [
    'protein_id', 'fasta_sample_id', 's1_sample_id', 's1_accession',
    's1_row', 'aa_length', 'sequence', 'fasta_description',
    'ecosystem_group', 'ncbi_taxonomy',
    'isolation_source_environmental_context', 'sample_materials',
    'latitude', 'longitude', 'depth_m', 'temperature_c',
    'related_srr_ids', 'total_bases_gb', 'source_doi_and_reference',
]


def fasta_records(path):
    with path.open(encoding='utf-8') as handle:
        header = None
        parts = []
        for raw_line in handle:
            line = raw_line.strip()
            if line.startswith('>'):
                if header is not None:
                    yield header, ''.join(parts)
                header = line[1:]
                parts = []
            elif line:
                parts.append(line)
        if header is not None:
            yield header, ''.join(parts)


def joined_rows():
    for header, sequence in fasta_records(fasta_path):
        protein_id, description = header.split(maxsplit=1)
        sample_id, _, gene_number = protein_id.rpartition('_')
        if not sample_id or not re.fullmatch(r'[0-9]{7}', gene_number):
            raise ValueError(f'蛋白 ID 格式异常：{protein_id}')
        if not sequence:
            raise ValueError(f'序列为空：{protein_id}')

        canonical = lookup_sample(sample_id)
        row_number, accession, metadata = samples[canonical]
        yield (
            protein_id, sample_id, canonical, accession, row_number,
            len(sequence), sequence, description, metadata[0], *metadata[2:],
        )

## 3. 导出并核验 CSV

逐行核对拼接结果。重复运行时直接核验已有 CSV。
更换输入数据后，可以调整 output_path 导出到新的文件。

In [4]:
def verify_csv(path):
    protein_ids = set()
    sample_ids = set()
    with path.open(encoding='utf-8-sig', newline='') as handle:
        reader = csv.reader(handle, strict=True)
        if next(reader, None) != columns:
            raise ValueError(f'CSV 列结构不同，请检查输出路径：{path.name}')

        for line_number, (saved, expected) in enumerate(
            zip_longest(reader, joined_rows()), start=2
        ):
            if saved is None or expected is None:
                raise ValueError(f'CSV 行数与 FASTA 不一致：第 {line_number} 行')
            expected_text = [str(value) if value is not None else '' for value in expected]
            if saved != expected_text:
                raise ValueError(f'CSV 内容与输入不一致：第 {line_number} 行')
            protein_id, _, canonical_id = saved[:3]
            if protein_id in protein_ids:
                raise ValueError(f'蛋白 ID 重复：{protein_id}')
            protein_ids.add(protein_id)
            sample_ids.add(canonical_id)

    if not protein_ids:
        raise ValueError('输入 FASTA 没有序列')
    return len(protein_ids), len(sample_ids)


def export_csv(path):
    if path.exists():
        return verify_csv(path)

    temporary_path = None
    try:
        with NamedTemporaryFile(
            mode='w', dir=path.parent, prefix=f'.{path.stem}_', suffix='.csv',
            encoding='utf-8-sig', newline='', delete=False,
        ) as handle:
            temporary_path = Path(handle.name)
            writer = csv.writer(handle)
            writer.writerow(columns)
            writer.writerows(joined_rows())

        counts = verify_csv(temporary_path)
        os.link(temporary_path, path)
        return counts
    finally:
        if temporary_path is not None:
            temporary_path.unlink(missing_ok=True)


protein_count, sample_count = export_csv(output_path)
print(f'{output_path.name} | {protein_count:,} 条蛋白序列 | {sample_count:,} 个样本')

meer1000M_cd90_DSGC_Table_S1.csv | 386,829 条蛋白序列 | 1,121 个样本


## 4. 配置 Swiss-Prot 蛋白比对

用 CD-HIT 代表序列 FASTA（`fasta_path`）查询本地 UniProtKB/Swiss-Prot，保留原始蛋白 ID（`qseqid`），可与前面 CSV 的 `protein_id` 关联。默认使用 12 个线程和 `--sensitive`，保留每条查询最多 10 个 E-value ≤ 1e-5 的目标。以下代码记录输入文件的 SHA-256、DIAMOND 版本和参数，便于重跑同一分析。

[DIAMOND 参数文档](https://github.com/bbuchfink/diamond/wiki/3.-Command-line-options)

In [5]:
from contextlib import suppress
from hashlib import sha256
import shlex
import shutil
import subprocess
from tempfile import TemporaryDirectory

swissprot_fasta = project / 'dataset/protein_annotation/uniprot_sprot.fasta'
diamond_dir = project / 'output/diamond'
threads = 12
sensitivity = '--sensitive'
evalue = 1e-5
max_targets = 10

bundled_diamond = project / 'dataset/MEER/MEER/annotation_2026-11-05/tools/diamond'
diamond = str(bundled_diamond) if os.access(bundled_diamond, os.X_OK) else shutil.which('diamond')
if diamond is None:
    raise FileNotFoundError('找不到 DIAMOND。请在当前环境安装 diamond，或设置 diamond 可执行文件路径。')
diamond_version = subprocess.check_output([diamond, 'version'], text=True).strip()

hit_columns = [
    'qseqid', 'sseqid', 'pident', 'length', 'qlen', 'slen',
    'qstart', 'qend', 'sstart', 'send', 'qcovhsp', 'scovhsp',
    'evalue', 'bitscore', 'stitle',
]
search_args = [
    '--threads', str(threads), sensitivity, '--evalue', str(evalue),
    '--max-target-seqs', str(max_targets), '--outfmt', '6', *hit_columns,
]


def file_sha256(path):
    digest = sha256()
    with path.open('rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


query_sha = file_sha256(fasta_path)
reference_sha = file_sha256(swissprot_fasta)
db_key = sha256(repr((diamond_version, reference_sha)).encode()).hexdigest()[:12]
run_key = sha256(repr((db_key, query_sha, search_args)).encode()).hexdigest()[:12]
db_path = diamond_dir / f'uniprot_sprot_{db_key}.dmnd'
hits_path = diamond_dir / f'meer1000M_cd90_vs_swissprot_{run_key}.tsv'

print(diamond_version)
print(f'查询 FASTA  {fasta_path.relative_to(project)}\nSHA-256    {query_sha}')
print(f'参考 FASTA  {swissprot_fasta.relative_to(project)}\nSHA-256    {reference_sha}')

diamond version 2.2.8
查询 FASTA  output/cdhit/meer1000M_cd90.pep.fa
SHA-256    0858ce53cedeccc9f70a97093ebee6cbf69f93e1ff13d2440e2dd237b564ae90
参考 FASTA  dataset/protein_annotation/uniprot_sprot.fasta
SHA-256    1f0a0d0a08d4c932ce91dca0b614fadfca87bd00931d7aad747c523f604d4f46


## 5. 建库并运行 DIAMOND blastp

在 `output/diamond/` 保存数据库和带表头的命中表。文件名随输入或相应配置变化。同一配置的重复运行复用已经完整写出的文件。

In [6]:
def run_diamond(command, stdout=None):
    print(shlex.join(command), flush=True)
    result = subprocess.run(command, stdout=stdout, stderr=subprocess.PIPE, text=True)
    print('\n'.join(result.stderr.splitlines()[-8:]))
    result.check_returncode()


diamond_dir.mkdir(parents=True, exist_ok=True)

makedb_args = [diamond, 'makedb', '--in', str(swissprot_fasta), '--threads', str(threads)]
blastp_args = [diamond, 'blastp', '--query', str(fasta_path), '--db', str(db_path), *search_args]

if not db_path.exists() or not hits_path.exists():
    with TemporaryDirectory(dir=diamond_dir, prefix='.diamond_') as work_dir:
        work_dir = Path(work_dir)
        if not db_path.exists():
            temp_db = work_dir / db_path.name
            makedb_cmd = [*makedb_args, '--db', str(temp_db)]
            run_diamond(makedb_cmd)
            with suppress(FileExistsError):
                os.link(temp_db, db_path)

        if not hits_path.exists():
            temp_hits = work_dir / hits_path.name
            blastp_cmd = [*blastp_args, '--tmpdir', str(work_dir)]
            # DIAMOND 的命中记录写到标准输出，直接接在表头后。
            with temp_hits.open('wb') as handle:
                handle.write(('\t'.join(hit_columns) + '\n').encode('utf-8'))
                handle.flush()
                run_diamond(blastp_cmd, stdout=handle)
            with suppress(FileExistsError):
                os.link(temp_hits, hits_path)
else:
    print('复用已有数据库和命中表。')

print(f'命中表  {hits_path.relative_to(project)}')

/Users/conwaysu/Documents/psychrophilic/Psychrophilic-protein-discovery/dataset/MEER/MEER/annotation_2026-11-05/tools/diamond makedb --in /Users/conwaysu/Documents/psychrophilic/Psychrophilic-protein-discovery/dataset/protein_annotation/uniprot_sprot.fasta --threads 12 --db /Users/conwaysu/Documents/psychrophilic/Psychrophilic-protein-discovery/output/diamond/.diamond_lzw1onld/uniprot_sprot_8e1f2b65e49c.dmnd
Database sequences  576073
  Database letters  209202719
     Database hash  77f6fc48d3f7dab51dba9b5dfb13c9d3
        Total time  0.803000s

/Users/conwaysu/Documents/psychrophilic/Psychrophilic-protein-discovery/dataset/MEER/MEER/annotation_2026-11-05/tools/diamond blastp --query /Users/conwaysu/Documents/psychrophilic/Psychrophilic-protein-discovery/output/cdhit/meer1000M_cd90.pep.fa --db /Users/conwaysu/Documents/psychrophilic/Psychrophilic-protein-discovery/output/diamond/uniprot_sprot_8e1f2b65e49c.dmnd --threads 12 --sensitive --evalue 1e-05 --max-target-seqs 10 --outfmt 6 qse

## 6. 核对命中数并预览结果

`sseqid` 含 Swiss-Prot accession（`sp|accession|entry` 的中间字段）。`pident` 是局部比对的一致性百分比，`qcovhsp` 和 `scovhsp` 分别表示该局部比对对查询和目标蛋白的覆盖百分比。解读命中时同时查看 E-value、一致性和覆盖度。相似性结果可用于提出功能或嗜冷性假设，后续仍需验证。

In [7]:
from IPython.display import display
import pandas as pd

matched_queries = set()
hit_count = 0
preview = []
with hits_path.open(encoding='utf-8', newline='') as handle:
    reader = csv.reader(handle, delimiter='\t', quoting=csv.QUOTE_NONE)
    if next(reader, None) != hit_columns:
        raise ValueError(f'命中表的列名与配置不符：{hits_path}')
    for row in reader:
        if len(row) != len(hit_columns):
            raise ValueError(f'命中表第 {reader.line_num} 行列数异常')
        matched_queries.add(row[0])
        hit_count += 1
        if len(preview) < 5:
            preview.append(row)

matched_count = len(matched_queries)
if matched_count > protein_count or hit_count > matched_count * max_targets:
    raise ValueError('命中计数与查询序列数或每条查询的目标数上限不一致')
print(f'有命中的查询  {matched_count:,} / {protein_count:,}（{matched_count / protein_count:.1%}）')
print(f'无命中的查询  {protein_count - matched_count:,}')
print(f'命中记录数    {hit_count:,}')

preview_columns = ['qseqid', 'sseqid', 'pident', 'qcovhsp', 'scovhsp', 'evalue', 'bitscore']
if preview:
    display(pd.DataFrame(preview, columns=hit_columns)[preview_columns])

有命中的查询  238,478 / 386,829（61.6%）
无命中的查询  148,351
命中记录数    1,997,487


,qseqid,sseqid,pident,qcovhsp,scovhsp,evalue,bitscore
0,FDZ031YE0-5_0000002,sp|P45394|YRBG_ECOLI,33.3,89.2,83.7,7.47e-48,166
1,FDZ031YE0-5_0000002,sp|Q0ZAI3|CAXA_ALKAM,36.8,82.4,78.1,1.50e-45,159
2,FDZ031YE0-5_0000002,sp|Q8TPA6|MAX1_METAC,29.9,96.1,92.6,3.76e-36,135
3,FDZ031YE0-5_0000002,sp|Q57556|Y091_METJA,35.6,81.4,78.1,5.14e-36,134
4,FDZ031YE0-5_0000003,sp|Q82C66|SYL_STRAW,56.4,100,66.8,5.55e-246,708
